
## Scenario: No Session Time Variation

**Description:** Same ICP keeps logging care sessions with the exact same start and end time, over and over.

In [ ]:
ENGINE_CATALOG = dbutils.widgets.get("ENGINE_CATALOG")
ENGINE_SCHEMA = dbutils.widgets.get("ENGINE_SCHEMA")

GRAPH_CATALOG = dbutils.widgets.get("GRAPH_CATALOG")
GRAPH_SCHEMA = dbutils.widgets.get("GRAPH_SCHEMA")

In [ ]:
%sql

DECLARE execDatetime TIMESTAMP = GETDATE();

In [ ]:
df = spark.sql(f"""
  SELECT 
    NOVEL_SCENARIO_ID 
  FROM
    {ENGINE_CATALOG}.{ENGINE_SCHEMA}.T_NOVEL_SCENARIO 
  WHERE 
    NOTEBOOK_NAME = 'Scenario_NoSessionTimeVariation'
""")

novelScenarioId = df.collect()[0][0]
print(f"Novel scenario ID: {novelScenarioId}")

## ICPs

In [ ]:
# Parameters for ICP invoices
icpInvoiceNumDaysLookback = 365
icpLatestCareNumDaysCutoff = 90

# Parameters for ICP sessions. The first session in each (claim, ICP) group
# has no prior session and is excluded from the comparison denominator.
icpMinComparableSessions = 20
icpMinIdenticalSessionFraction = 0.95

In [ ]:
spark.sql(f"""
SELECT CLAIM_ID, CLAIM_NUMBER FROM {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_CLAIM
WHERE CLAIM_STATUS_CODE IN ('Active', 'ASWP', 'Benefit Period', 'Qualification Period')
""").createOrReplaceTempView("active_claims")

In [ ]:
# Provider qualification is kept at (claim, provider) grain, matching
# Scenario_NoInvoiceVariation. TOTAL_CHARGE_AMT / TOTAL_PAY_AMT are carried
# through to the detail table on every rule per repo convention.
spark.sql(f"""
SELECT
  i.CLAIM_ID,
  c.CLAIM_NUMBER,
  p.RES_PERSON_ID AS PROVIDER_ID,
  COUNT(DISTINCT i.NORM_INVOICE_ID) AS NUM_INVOICES,
  SUM(i.INVOICE_CHARGE_AMT) AS TOTAL_CHARGE_AMT,
  SUM(i.INVOICE_PAY_AMT) AS TOTAL_PAY_AMT,
  MAX(i.INVOICE_SERVICE_END_DATE) AS CARE_END_DATE
FROM
  {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_INVOICE i
JOIN
  {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON_INVOICE_CROSSWALK p
  ON i.NORM_INVOICE_ID = p.NORM_INVOICE_ID
  AND p.EDGE_NAME = 'PROVIDED_CARE_ON_INVOICE'
JOIN
  active_claims c
  ON i.CLAIM_ID = c.CLAIM_ID
WHERE
  i.INVOICE_SERVICE_END_DATE >= DATE_SUB(GETDATE(), {icpInvoiceNumDaysLookback})
GROUP BY
  i.CLAIM_ID,
  c.CLAIM_NUMBER,
  p.RES_PERSON_ID
HAVING
  MAX(i.INVOICE_SERVICE_END_DATE) >= DATE_SUB(GETDATE(), {icpLatestCareNumDaysCutoff})
""").createOrReplaceTempView("icp_invoice_summary")

In [ ]:
spark.sql(f"""
SELECT DISTINCT
  cs.CLAIM_ID,
  rpcs.RES_PERSON_ID,
  cs.NORM_CARE_SESSION_ID,
  cs.SESSION_START_TS,
  DATE_FORMAT(cs.SESSION_START_TS, 'HH:mm') AS START_TIME,
  DATE_FORMAT(cs.SESSION_END_TS, 'HH:mm') AS END_TIME
FROM
  {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_CARE_SESSION cs
JOIN
  {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON_CARE_SESSION_CROSSWALK rpcs
  ON rpcs.NORM_CARE_SESSION_ID = cs.NORM_CARE_SESSION_ID
  AND rpcs.EDGE_NAME = 'PROVIDED_CARE_ON_SESSION'
JOIN
  active_claims c
  ON cs.CLAIM_ID = c.CLAIM_ID
WHERE
  cs.SESSION_START_TS IS NOT NULL
  AND cs.SESSION_END_TS IS NOT NULL
  AND cs.SESSION_START_TS >= DATE_SUB(GETDATE(), {icpInvoiceNumDaysLookback})
""").createOrReplaceTempView("icp_provider_sessions")

In [ ]:
spark.sql(f"""
SELECT
  CLAIM_ID,
  RES_PERSON_ID AS PROVIDER_ID,
  NORM_CARE_SESSION_ID,
  START_TIME,
  END_TIME,
  LAG(START_TIME) OVER (PARTITION BY CLAIM_ID, RES_PERSON_ID ORDER BY SESSION_START_TS, NORM_CARE_SESSION_ID) AS PRIOR_START_TIME,
  LAG(END_TIME) OVER (PARTITION BY CLAIM_ID, RES_PERSON_ID ORDER BY SESSION_START_TS, NORM_CARE_SESSION_ID) AS PRIOR_END_TIME
FROM
  icp_provider_sessions
""").createOrReplaceTempView("icp_session_comparisons")

In [ ]:
spark.sql(f"""
SELECT
  CLAIM_ID,
  PROVIDER_ID,
  COUNT(*) AS COMPARABLE_SESSION_COUNT,
  SUM(CASE WHEN START_TIME = PRIOR_START_TIME AND END_TIME = PRIOR_END_TIME THEN 1 ELSE 0 END) AS IDENTICAL_SESSION_COUNT,
  TRY_DIVIDE(
    SUM(CASE WHEN START_TIME = PRIOR_START_TIME AND END_TIME = PRIOR_END_TIME THEN 1 ELSE 0 END),
    COUNT(*)
  ) AS FRACTION_IDENTICAL_SESSIONS
FROM
  icp_session_comparisons
WHERE
  PRIOR_START_TIME IS NOT NULL
  AND PRIOR_END_TIME IS NOT NULL
GROUP BY
  CLAIM_ID,
  PROVIDER_ID
HAVING
  COUNT(*) >= {icpMinComparableSessions}
  AND TRY_DIVIDE(
    SUM(CASE WHEN START_TIME = PRIOR_START_TIME AND END_TIME = PRIOR_END_TIME THEN 1 ELSE 0 END),
    COUNT(*)
  ) >= {icpMinIdenticalSessionFraction}
""").createOrReplaceTempView("icp_session_summary")

In [ ]:
# Most-common repeated (start, end) pair per (claim, provider) -- the
# schedule that is being copy-pasted across sessions.
spark.sql(f"""
SELECT CLAIM_ID, PROVIDER_ID, START_TIME, END_TIME
FROM (
  SELECT
    CLAIM_ID,
    PROVIDER_ID,
    START_TIME,
    END_TIME,
    ROW_NUMBER() OVER (
      PARTITION BY CLAIM_ID, PROVIDER_ID
      ORDER BY COUNT(*) DESC, START_TIME, END_TIME
    ) AS RN
  FROM icp_session_comparisons
  GROUP BY CLAIM_ID, PROVIDER_ID, START_TIME, END_TIME
)
WHERE RN = 1
""").createOrReplaceTempView("icp_session_time_mode")

In [ ]:
spark.sql(f"""
SELECT
  i.CLAIM_ID,
  i.CLAIM_NUMBER,
  i.PROVIDER_ID,
  CONCAT(p.FIRST_NAME, ' ', p.LAST_NAME) AS PROVIDER_NAME,
  'ICP' AS PROVIDER_TYPE,
  i.NUM_INVOICES,
  i.TOTAL_CHARGE_AMT,
  i.TOTAL_PAY_AMT,
  i.CARE_END_DATE,
  s.COMPARABLE_SESSION_COUNT,
  s.IDENTICAL_SESSION_COUNT,
  s.FRACTION_IDENTICAL_SESSIONS,
  m.START_TIME AS MOST_COMMON_SESSION_START_TIME,
  m.END_TIME AS MOST_COMMON_SESSION_END_TIME
FROM
  icp_invoice_summary i
JOIN
  icp_session_summary s
  ON i.CLAIM_ID = s.CLAIM_ID
  AND i.PROVIDER_ID = s.PROVIDER_ID
JOIN
  icp_session_time_mode m
  ON i.CLAIM_ID = m.CLAIM_ID
  AND i.PROVIDER_ID = m.PROVIDER_ID
JOIN
  {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON p
  ON i.PROVIDER_ID = p.RES_PERSON_ID
""").createOrReplaceTempView("flagged_claims")

In [ ]:
spark.sql(f"""
    INSERT INTO {ENGINE_CATALOG}.{ENGINE_SCHEMA}.T_SCENARIO_NO_SESSION_TIME_VARIATION_DETAIL
    SELECT 
      *,
      GETDATE() AS FEATURE_DATETIME
    FROM flagged_claims;
""")